In [5]:
import pandas as pd

df = pd.read_excel("../data/raw/aqi/WORLI 24-26.xlsx", header=None)

for i in range(40):
    print(i, df.iloc[i].tolist())

0 ['CENTRAL POLLUTION CONTROL BOARD', nan, nan, nan, nan, nan, nan]
1 ['CONTINUOUS AMBIENT AIR QUALITY', nan, nan, nan, nan, nan, nan]
2 ['Date: Tuesday, Jun 09 2026', nan, nan, nan, nan, nan, nan]
3 ['Time: 12:22:14 PM', nan, nan, nan, nan, nan, nan]
4 ['State', 'Maharashtra', nan, nan, nan, nan, nan]
5 ['City', 'Mumbai', nan, nan, nan, nan, nan]
6 ['Station', 'Worli, Mumbai -MPCB', nan, nan, nan, nan, nan]
7 ['Parameter', 'PM2.5,PM10,Ozone,SO2,NO2', nan, nan, nan, nan, nan]
8 ['AvgPeriod', '1 Hours', nan, nan, nan, nan, nan]
9 ['From', '01-01-2024T00:00:00Z 00:00', nan, nan, nan, nan, nan]
10 ['To', '31-05-2026T00:00:59Z 00:00', nan, nan, nan, nan, nan]
11 [nan, nan, nan, nan, nan, nan, nan]
12 ['Worli, Mumbai -MPCB', nan, nan, nan, nan, nan, nan]
13 ['Prescribed Standards', ' ', '0-60', '0-100', '0-180', '0-80', '0-80']
14 ['Exceeding Standards', ' ', nan, nan, nan, nan, nan]
15 ['Remarks', nan, nan, nan, nan, nan, nan]
16 ['From Date', 'To Date', 'PM2.5', 'PM10', 'Ozone', 'SO2', 'N

In [1]:
from pathlib import Path
import pandas as pd

# ============================================================
# Paths
# ============================================================

AQI_FOLDER = Path("../data/raw/aqi")
OUTPUT = Path("../data/processed/merged_aqi.csv")

# ============================================================
# Get all AQI Excel files
# ============================================================

files = sorted(
    f for f in AQI_FOLDER.glob("*.xlsx")
    if not f.name.startswith("~$")
)

if len(files) == 0:
    raise FileNotFoundError(f"No Excel files found in: {AQI_FOLDER}")

dataframes = []

print("=" * 60)
print("MERGING AQI DATASETS")
print("=" * 60)

# ============================================================
# Process each file
# ============================================================

for file in files:

    print(f"Processing: {file.name}")

    try:
        # --------------------------------------------------------
        # Read metadata section
        # --------------------------------------------------------
        metadata = pd.read_excel(file, header=None)

        # Station name is stored in row 6, column 1
        station_name = metadata.iloc[6, 1]

        # --------------------------------------------------------
        # Read actual AQI table
        # --------------------------------------------------------
        df = pd.read_excel(file, header=16)

        # --------------------------------------------------------
        # Remove completely empty rows & columns
        # --------------------------------------------------------
        df.dropna(how="all", inplace=True)
        df.dropna(axis=1, how="all", inplace=True)

        # --------------------------------------------------------
        # Clean column names
        # --------------------------------------------------------
        df.columns = df.columns.str.strip()

        # --------------------------------------------------------
        # Convert dates
        # --------------------------------------------------------
        if "From Date" in df.columns:
            df["From Date"] = pd.to_datetime(
                df["From Date"],
                errors="coerce"
            )

        if "To Date" in df.columns:
            df["To Date"] = pd.to_datetime(
                df["To Date"],
                errors="coerce"
            )

        # --------------------------------------------------------
        # Add metadata
        # --------------------------------------------------------
        df["Station"] = station_name
        df["Source_File"] = file.stem

        dataframes.append(df)

    except Exception as e:
        print(f"Error processing {file.name}")
        print(e)

# ============================================================
# Merge all files
# ============================================================

merged = pd.concat(dataframes, ignore_index=True)

# ============================================================
# Save merged dataset
# ============================================================

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

merged.to_csv(OUTPUT, index=False)

# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 60)
print("AQI FILES MERGED SUCCESSFULLY")
print("=" * 60)

print(f"Files Processed : {len(files)}")
print(f"Rows            : {merged.shape[0]}")
print(f"Columns         : {merged.shape[1]}")

print("\nColumns:")
for col in merged.columns:
    print(f" - {col}")

print(f"\nSaved to: {OUTPUT}")
print("=" * 60)

MERGING AQI DATASETS
Processing: AIRPORT 24-26.xlsx
Processing: AMBERNATH 24-26.xlsx
Processing: ANDHERI 24-26.xlsx
Processing: BADLAPUR 24-26.xlsx
Processing: BELAPUR 24-26.xlsx
Processing: BHANDUP 24-26.xlsx
Processing: BHAYANDAR 24-26.xlsx
Processing: BHIWANDI 24-26.xlsx
Processing: BKC 24-26.xlsx
Processing: BOISAR 24-26.xlsx
Processing: BORIVALI 24-26.xlsx
Processing: BYCULLA 24-26.xlsx
Processing: CHEMBUR 24-26.xlsx
Processing: COLABA 24-26.xlsx
Processing: DEONAR 24-26.xlsx
Processing: GHATKOPAR 24-26.xlsx
Processing: KALUNAGAR D 24-26.xlsx
Processing: KANDIVALI 24-26.xlsx
Processing: KASARVADAVALI 24-26.xlsx
Processing: KHADAKPADA KALYAN 24-26.xlsx
Processing: MAHAPE 24-26.xlsx
Processing: MALAD 24-26.xlsx
Processing: MAZGAON 24-26.xlsx
Processing: MULUND 24-26.xlsx
Processing: NERUL 24-26.xlsx
Processing: PIMPALESHWAR KALYAN 24-26.xlsx
Processing: POWAI 24-26.xlsx
Processing: SEWRI 24-26.xlsx
Processing: SIDDHIVINAYAK UN 24-26.xlsx
Processing: SION 24-26.xlsx
Processing: SONPA